# Phase 1 — Proof of Concept & Initial Data Collection

## Overview
This is the starting point of the AI-Powered Environmental Health Literature Extraction project. The goal of Phase 1 is straightforward: prove that it is possible to feed a scientific paper's text content to a large language model and get back structured, meaningful information about what was studied. This phase also establishes the first data collection mechanism by querying Google Scholar for VA Normative Aging Study papers and exporting their metadata to CSV.

---

## Context: Why This Project Exists
Environmental health researchers at Dr. Yike Shen's lab are building CONNECT (Cohort-Oriented Network kNowledgebase for Enhanced CollaboraTion), a knowledge graph linking environmental exposures to health outcomes across longitudinal cohort studies. Populating that database manually — by reading and coding every paper — is time-consuming and hard to scale. This project explores whether GPT can automate that extraction step reliably enough to be useful in a real research workflow.

---

## What This Notebook Does, Step by Step

### Setup (Cells 1–2)
- Installs `openai` (v1.77.0) and `PyMuPDF` (`fitz`) for PDF text extraction.
- Sets the OpenAI API key directly in code.  
  ⚠️ **Note:** The API key is hard-coded in this notebook. It has been replaced with a placeholder for GitHub. Store your key in a `.env` file and load it with `python-dotenv` before running.

### Single-Paper Proof of Concept (Cells 3–5)
- Uses `fitz.open()` to extract raw text from a local PDF of one of Dr. Shen's published papers: *"Assessing Consumer Buy and Pay Preferences for Labeled Food Products with Statistical and Machine Learning Methods"* (Shen et al., Michigan State University).
- Three successive attempts are made to extract title, authors, and abstract from the PDF:
  - **Cell 3:** Extracts first 20 lines for inspection — establishes the raw text structure.
  - **Cell 4:** Positions-based extraction (title = lines[1]+lines[2], authors = lines[3:10], abstract = text between "ABSTRACT" and the next all-caps header). Works for this paper's layout.
  - **Cell 5:** Regex-based extraction targeting known strings in the document. Returns title, authors, and abstract.
- All three cells output to the terminal — no CSV is saved at this stage. The goal is just to confirm the PDF-to-text pipeline works.

### Unzipping and Exploring the Test Corpus (Cells 6–14)
- Unzips `TestPapers.zip` into `./test_papers/`, a folder of 14 PDFs from Dr. Shen's publication history including papers from *Environmental Science & Technology*, *Journal of Hazardous Materials*, *Environmental International*, *eClinicalMedicine*, and others.
- Iterates through all 14 PDFs, extracts title/authors/abstract using the `extract_clean_pdf_info()` function, and prints results. Notes that some PDFs (e.g., Prada et al.) fail to return clean abstracts due to layout differences between journals.
- **Cell 14** adds a keyword filter: only papers whose abstract contains both "exposure" and "outcome" language are kept. **Result: 0 papers pass the filter** from the test corpus — Dr. Shen's own lab papers are not written in exposure/outcome framing because they cover a broader range of topics. This finding motivates the pivot to the VA Normative Aging Study corpus, which is explicitly built around that framework.

### Domain Definitions via GPT (Cells 15–17)
- Before building an extraction system, the team uses GPT-3.5-turbo (Cell 15) and GPT-4 (Cell 16–17) to generate definitions of core environmental health terms: *exposure*, *outcome*, *biomarker*, *chemical exposure*, *physical exposure*, *psychosocial exposure*, and *behavioral exposure*.
- The same question ("What is chemical exposure in X?") is asked across 7 different disciplinary framings (environmental health, health sciences, biomedical sciences, public health, occupational health, toxicology, medicine) to explore whether the domain framing changes the answer. It does not significantly — confirming that GPT has stable, domain-appropriate knowledge of these concepts.
- These outputs inform the taxonomy design used in later phases.

### Google Scholar Data Collection (Cells 18–20)
- Installs the `scholarly` library to programmatically query Google Scholar.
- Defines a list of 19 paper titles from the VA Normative Aging Study (sourced from the ground truth CSV `CohortNetwork_ES&T_SI_B_Main(Table S1).csv`).
- Queries Google Scholar for each title, retrieving: abstract, authors, year, journal, and URL.
- Deduplicates results and exports to `layer1_google_scholar_abstracts.csv`.
- **Result:** Several papers fail with `Cannot Fetch from Google Scholar` (rate limiting or bot detection). Those that succeed return usable metadata.

---

## Key Outputs
- `layer1_google_scholar_abstracts.csv` — metadata (title, abstract, authors, year, URL) for successfully retrieved VA Normative Aging Study papers

## Known Limitations
- Google Scholar has aggressive rate limiting — not all 19 papers were retrieved successfully.
- The abstract extraction from PDFs is layout-dependent and fails on papers with non-standard formatting.
- This phase does not yet attempt taxonomy-level extraction; it only confirms the pipeline concept is viable.

## Dependencies
```
openai
PyMuPDF (fitz)
scholarly
pandas
```
⚠️ Replace `API_KEY = "YOUR_OPENAI_API_KEY_HERE"` with `os.getenv("OPENAI_API_KEY")` and store your key in a `.env` file.

In [2]:
!pip install openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.0/662.0 kB 9.5 MB/s eta 0:00:00


In [4]:
import openai

openai.api_key = "YOUR_OPENAI_API_KEY_HERE"  


In [6]:
!pip install PyMuPDF


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.6/18.6 MB 31.1 MB/s eta 0:00:00a 0:00:01


In [10]:
import fitz  

def extract_pdf_info(file_path):
    with fitz.open(file_path) as doc:
        text = ""
        for page in doc:
            text += page.get_text()

    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    # DEBUG: See first 20 lines
    print("===== First 20 Lines =====")
    for i, line in enumerate(lines[:20]):
        print(f"{i}: {line}")
    print("==========================")

    # Title: usually first line
    title = lines[0] if len(lines) > 0 else "N/A"

    # Authors: usually after title, but may require better logic
    authors = ""
    for i in range(1, 6):  # Check next few lines
        if "abstract" not in lines[i].lower():
            authors += lines[i] + " "
        else:
            break
    authors = authors.strip()

    # Abstract
    abstract = ""
    for i, line in enumerate(lines):
        if "abstract" in line.lower():
            # Combine lines after abstract until a likely section heading
            abstract_lines = []
            for j in range(i + 1, len(lines)):
                if lines[j].lower() in ["introduction", "keywords"] or lines[j].endswith(":"):
                    break
                abstract_lines.append(lines[j])
            abstract = " ".join(abstract_lines).strip()
            break

    return title, authors, abstract

# Example
file_path = "Shen et al 2021_JFP.pdf"
title, authors, abstract = extract_pdf_info(file_path)
print("\nTitle:", title)
print("Authors:", authors)
print("Abstract:", abstract)


===== First 20 Lines =====
0: Research Paper
1: Assessing Consumer Buy and Pay Preferences for Labeled Food
2: Products with Statistical and Machine Learning Methods
3: YIKE SHEN
4: https://orcid.org/0000-0001-8480-8258,1,2,3 JOSEPH A. HAMM,2,4 FENG GAO,1,3,5 ELLIOT T. RYSER,6 AND
5: WEI ZHANG
6: https://orcid.org/0000-0002-2937-17321,2*
7: 1Department of Plant, Soil and Microbial Sciences, 2Environmental Science and Policy Program, 3Institute for Integrative Toxicology, 4School of Criminal
8: Justice, 5Department of Computational Mathematics, Science and Engineering, and 6Department of Food Science and Human Nutrition, Michigan State
9: University, East Lansing, Michigan 48824, USA
10: MS 20-486: Received 17 December 2020/Accepted 13 May 2021/Published Online 13 May 2021
11: ABSTRACT
12: Food labeling is one approach to encourage safe, healthy, and sustainable dietary practices. Consumer buy and pay
13: preferences for specially labeled food products (e.g., U.S. Department of Agricult

In [12]:
import fitz  # PyMuPDF

def extract_pdf_info(file_path):
    with fitz.open(file_path) as doc:
        text = ""
        for page in doc:
            text += page.get_text()

    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    # Extract title from known position
    title = f"{lines[1]} {lines[2]}" if len(lines) >= 3 else lines[0]

    # Extract authors from known range
    authors = " ".join(lines[3:10])

    # Extract abstract between "ABSTRACT" and "HIGHLIGHTS"
    abstract = ""
    try:
        abstract_start = lines.index("ABSTRACT")
        # Collect until "HIGHLIGHTS" or until a line in all caps (section heading)
        for i in range(abstract_start + 1, len(lines)):
            if lines[i].isupper() and lines[i] != "ABSTRACT":
                break
            abstract += lines[i] + " "
    except ValueError:
        abstract = "Abstract not found."

    return title.strip(), authors.strip(), abstract.strip()

# Example usage
file_path = "Shen et al 2021_JFP.pdf"
title, authors, abstract = extract_pdf_info(file_path)
print("\nTitle:", title)
print("\nAuthors:", authors)
print("\nAbstract:", abstract)



Title: Assessing Consumer Buy and Pay Preferences for Labeled Food Products with Statistical and Machine Learning Methods

Authors: YIKE SHEN https://orcid.org/0000-0001-8480-8258,1,2,3 JOSEPH A. HAMM,2,4 FENG GAO,1,3,5 ELLIOT T. RYSER,6 AND WEI ZHANG https://orcid.org/0000-0002-2937-17321,2* 1Department of Plant, Soil and Microbial Sciences, 2Environmental Science and Policy Program, 3Institute for Integrative Toxicology, 4School of Criminal Justice, 5Department of Computational Mathematics, Science and Engineering, and 6Department of Food Science and Human Nutrition, Michigan State University, East Lansing, Michigan 48824, USA

Abstract: Food labeling is one approach to encourage safe, healthy, and sustainable dietary practices. Consumer buy and pay preferences for specially labeled food products (e.g., U.S. Department of Agriculture organic, raised without antibiotics, and locally raised) may promote the adoption of associated production practices by food producers. Thus, it is imp

In [96]:


def extract_paper_info(pdf_path):
    doc = fitz.open(pdf_path)
    text = "\n".join([page.get_text() for page in doc])
    doc.close()

    # Try to extract the title
    title_match = re.search(r"(Assessing Consumer Buy and Pay Preferences.*?)\n", text, re.DOTALL)
    title = title_match.group(1).strip() if title_match else "Title not found"

    # Extract authors - find the block following the title
    author_block_match = re.search(r"(?<=\n)(YIKE SHEN.*?WEI ZHANG.*?)\n\n", text, re.DOTALL)
    authors = author_block_match.group(1).strip() if author_block_match else "Authors not found"

    # Extract abstract
    abstract_match = re.search(r"ABSTRACT\s*\n(.+?)\n\n", text, re.DOTALL)
    abstract = abstract_match.group(1).strip() if abstract_match else "Abstract not found"

    return {
        "title": title,
        "authors": authors,
        "abstract": abstract
    }

# Example usage
paper_info = extract_paper_info("Shen et al 2021_JFP.pdf")
for key, value in paper_info.items():
    print(f"{key.upper()}:\n{value}\n")


TITLE:
Assessing Consumer Buy and Pay Preferences for Labeled Food

AUTHORS:
YIKE SHEN
https://orcid.org/0000-0001-8480-8258,1,2,3 JOSEPH A. HAMM,2,4 FENG GAO,1,3,5 ELLIOT T. RYSER,6 AND
WEI ZHANG
https://orcid.org/0000-0002-2937-17321,2*
1Department of Plant, Soil and Microbial Sciences, 2Environmental Science and Policy Program, 3Institute for Integrative Toxicology, 4School of Criminal
Justice, 5Department of Computational Mathematics, Science and Engineering, and 6Department of Food Science and Human Nutrition, Michigan State
University, East Lansing, Michigan 48824, USA
MS 20-486: Received 17 December 2020/Accepted 13 May 2021/Published Online 13 May 2021
ABSTRACT
Food labeling is one approach to encourage safe, healthy, and sustainable dietary practices. Consumer buy and pay
preferences for specially labeled food products (e.g., U.S. Department of Agriculture organic, raised without antibiotics, and
locally raised) may promote the adoption of associated production practices by fo

In [130]:
import zipfile
import os

zip_file = "TestPapers.zip"
extract_path = "./test_papers"

# Extract the zip file
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped files to:", extract_path)


Unzipped files to: ./test_papers


In [132]:
pdf_files = [f for f in os.listdir(extract_path) if f.lower().endswith(".pdf")]
print("Found PDF files:", pdf_files)


Found PDF files: []


In [134]:
import os

for root, dirs, files in os.walk("./test_papers"):
    print(f" Directory: {root}")
    for file in files:
        print("  ", file)


 Directory: ./test_papers
 Directory: ./test_papers/TestPapers
   Prada et al., 2023_eClinMed.pdf
   Gao & Shen et al., 2022_JHM.pdf
   Gao et al., ES&T_2021.pdf
   Gao et al., EI_2022.pdf
   Shen et al., EHP_2022.pdf
   Campana et al., 2022.pdf
   Shen et al 2021_JFP.pdf
   Shen et al., 2021_STOTEN.pdf
   Shen et al., JHM_2022.pdf
   Shen et al., 2019_EI.pdf
   Shen et al., 2023_ES&T.pdf
   Chen et al., 2023_Environmental Microbiology.pdf
   Shen et al., 2021_CHEM.pdf
   Laue & Shen et al., 2022.pdf
 Directory: ./test_papers/__MACOSX
   ._TestPapers
 Directory: ./test_papers/__MACOSX/TestPapers
   ._Shen et al., 2023_ES&T.pdf
   ._Shen et al., 2019_EI.pdf
   ._Gao et al., ES&T_2021.pdf
   ._Shen et al., 2021_CHEM.pdf
   ._Shen et al., EHP_2022.pdf
   ._Shen et al., JHM_2022.pdf
   ._Shen et al., 2021_STOTEN.pdf
   ._Gao et al., EI_2022.pdf
   ._Shen et al 2021_JFP.pdf
   ._Prada et al., 2023_eClinMed.pdf
   ._Campana et al., 2022.pdf
   ._Laue & Shen et al., 2022.pdf
   ._Gao & Shen e

In [136]:
pdf_files = []
for root, dirs, files in os.walk("./test_papers"):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_files.append(os.path.join(root, file))

print(" Found PDF files:")
for f in pdf_files:
    print(" -", f)


 Found PDF files:
 - ./test_papers/TestPapers/Prada et al., 2023_eClinMed.pdf
 - ./test_papers/TestPapers/Gao & Shen et al., 2022_JHM.pdf
 - ./test_papers/TestPapers/Gao et al., ES&T_2021.pdf
 - ./test_papers/TestPapers/Gao et al., EI_2022.pdf
 - ./test_papers/TestPapers/Shen et al., EHP_2022.pdf
 - ./test_papers/TestPapers/Campana et al., 2022.pdf
 - ./test_papers/TestPapers/Shen et al 2021_JFP.pdf
 - ./test_papers/TestPapers/Shen et al., 2021_STOTEN.pdf
 - ./test_papers/TestPapers/Shen et al., JHM_2022.pdf
 - ./test_papers/TestPapers/Shen et al., 2019_EI.pdf
 - ./test_papers/TestPapers/Shen et al., 2023_ES&T.pdf
 - ./test_papers/TestPapers/Chen et al., 2023_Environmental Microbiology.pdf
 - ./test_papers/TestPapers/Shen et al., 2021_CHEM.pdf
 - ./test_papers/TestPapers/Laue & Shen et al., 2022.pdf
 - ./test_papers/__MACOSX/TestPapers/._Shen et al., 2023_ES&T.pdf
 - ./test_papers/__MACOSX/TestPapers/._Shen et al., 2019_EI.pdf
 - ./test_papers/__MACOSX/TestPapers/._Gao et al., ES&T_202

In [138]:
for full_path in pdf_files:
    title, authors, abstract = extract_pdf_info(full_path)
    
    print(f"\n📄 {os.path.basename(full_path)}")
    print("Title:", title)
    print("Authors:", authors)
    print("Abstract:", abstract[:300] + "..." if len(abstract) > 300 else abstract)
    print("-" * 80)



📄 Prada et al., 2023_eClinMed.pdf
Title: Air pollution and decreased bone mineral density among
Authors: Women’s Health Initiative participants Diddier Prada,a,b,h,∗Carolyn J. Crandall,c,i Allison Kupsco,a,j Marianthi-Anna Kioumourtzoglou,a,k James D. Stewart,d,l Duanping Liao,e,m Jeff D. Yanosky,e,m Andrea Ramirez,b,n Jean Wactawski-Wende,f,o Yike Shen,a,h Gary Miller,a,p Iuliana Ionita-Laza,a,q Eric A. Whitsel,g,l and Andrea A. Baccarellia,h aDepartment of Environmental Health Science, Mailman School of Public Health, Columbia University, New York, NY, USA
Abstract: Abstract not found.
--------------------------------------------------------------------------------

📄 Gao & Shen et al., 2022_JHM.pdf
Title: Predicting crop root concentration factors of organic contaminants with
Authors: machine learning models Feng Gao a,1, Yike Shen b,1, J. Brett Sallach c, Hui Li d, Wei Zhang d, Yuanbo Li e,*, Cun Liu f,* a Department of Genetics, School of Medicine, Yale University, New Haven, CT 

FileDataError: Failed to open file './test_papers/__MACOSX/TestPapers/._Shen et al., 2023_ES&T.pdf'.

In [140]:
pdf_files = []
for root, dirs, files in os.walk("./test_papers"):
    for file in files:
        if file.lower().endswith(".pdf") and not file.startswith("._"):
            pdf_files.append(os.path.join(root, file))

print(f"✅ Found {len(pdf_files)} actual PDF files:")
for f in pdf_files:
    print(" -", f)


✅ Found 14 actual PDF files:
 - ./test_papers/TestPapers/Prada et al., 2023_eClinMed.pdf
 - ./test_papers/TestPapers/Gao & Shen et al., 2022_JHM.pdf
 - ./test_papers/TestPapers/Gao et al., ES&T_2021.pdf
 - ./test_papers/TestPapers/Gao et al., EI_2022.pdf
 - ./test_papers/TestPapers/Shen et al., EHP_2022.pdf
 - ./test_papers/TestPapers/Campana et al., 2022.pdf
 - ./test_papers/TestPapers/Shen et al 2021_JFP.pdf
 - ./test_papers/TestPapers/Shen et al., 2021_STOTEN.pdf
 - ./test_papers/TestPapers/Shen et al., JHM_2022.pdf
 - ./test_papers/TestPapers/Shen et al., 2019_EI.pdf
 - ./test_papers/TestPapers/Shen et al., 2023_ES&T.pdf
 - ./test_papers/TestPapers/Chen et al., 2023_Environmental Microbiology.pdf
 - ./test_papers/TestPapers/Shen et al., 2021_CHEM.pdf
 - ./test_papers/TestPapers/Laue & Shen et al., 2022.pdf


In [148]:
for full_path in pdf_files:
    try:
        title, authors, abstract = extract_pdf_info(full_path)

        print(f"\n {os.path.basename(full_path)}")
        print("Title:", title)
        print("Authors:", authors)
        print("Abstract:", abstract[:300] + "..." if len(abstract) > 300 else abstract)
        print("-" * 80)
    except Exception as e:
        print(f"\n Failed to read {os.path.basename(full_path)}: {e}")



 Prada et al., 2023_eClinMed.pdf
Title: Air pollution and decreased bone mineral density among
Authors: Women’s Health Initiative participants Diddier Prada,a,b,h,∗Carolyn J. Crandall,c,i Allison Kupsco,a,j Marianthi-Anna Kioumourtzoglou,a,k James D. Stewart,d,l Duanping Liao,e,m Jeff D. Yanosky,e,m Andrea Ramirez,b,n Jean Wactawski-Wende,f,o Yike Shen,a,h Gary Miller,a,p Iuliana Ionita-Laza,a,q Eric A. Whitsel,g,l and Andrea A. Baccarellia,h aDepartment of Environmental Health Science, Mailman School of Public Health, Columbia University, New York, NY, USA
Abstract: Abstract not found.
--------------------------------------------------------------------------------

 Gao & Shen et al., 2022_JHM.pdf
Title: Predicting crop root concentration factors of organic contaminants with
Authors: machine learning models Feng Gao a,1, Yike Shen b,1, J. Brett Sallach c, Hui Li d, Wei Zhang d, Yuanbo Li e,*, Cun Liu f,* a Department of Genetics, School of Medicine, Yale University, New Haven, CT 06

In [154]:
def extract_clean_pdf_info(pdf_path):
    doc = fitz.open(pdf_path)
    all_text = []
    for i, page in enumerate(doc):
        all_text += page.get_text().splitlines()
        if i == 1:  # First two pages
            break
    doc.close()

    lines = [line.strip() for line in all_text if line.strip()]

    # Title: first long line that isn't a header
    title = "Title not found"
    for line in lines:
        if 40 < len(line) < 200 and not line.lower().startswith(("available online", "journal", "copyright", "doi", "contents lists", "article history", "highlights", "0304-", "environmental", "plos", "sciencedirect")):
            title = line
            break

    # Authors: first line with many commas and "and"
    authors = "Authors not found"
    for line in lines:
        if len(re.findall(r",", line)) >= 2 and re.search(r"\band\b", line, re.IGNORECASE):
            authors = line
            break

    # Abstract: everything between 'abstract' and the next all-caps or intro-like heading
    abstract = "Abstract not found"
    try:
        idx = next(i for i, line in enumerate(lines) if "abstract" in line.lower())
        abstract_lines = []
        for line in lines[idx + 1:]:
            if line.isupper() or line.lower().startswith("introduction"):
                break
            abstract_lines.append(line)
        if abstract_lines:
            abstract = " ".join(abstract_lines)
    except StopIteration:
        pass

    return title, authors, abstract

for full_path in pdf_files:
    try:
        title, authors, abstract = extract_clean_pdf_info(full_path)

        print(f"\n {os.path.basename(full_path)}")
        print(f"{'Title:':<10} {title}")
        print(f"{'Authors:':<10} {authors}")
        print(f"{'Abstract:':<10} {abstract[:300]}..." if len(abstract) > 300 else f"{'Abstract:':<10} {abstract}")
        print("-" * 100)

    except Exception as e:
        print(f"\n Failed to read {os.path.basename(full_path)}: {e}")



 Prada et al., 2023_eClinMed.pdf
Title:     Air pollution and decreased bone mineral density among
Authors:   Jeff D. Yanosky,e,m Andrea Ramirez,b,n Jean Wactawski-Wende,f,o Yike Shen,a,h Gary Miller,a,p Iuliana Ionita-Laza,a,q Eric A. Whitsel,g,l and
Abstract:  Abstract not found
----------------------------------------------------------------------------------------------------

 Gao & Shen et al., 2022_JHM.pdf
Title:     Predicting crop root concentration factors of organic contaminants with
Authors:   c Department of Environment and Geography, University of York, Heslington, York YO10 5NG, United Kingdom
Abstract:  Abstract not found
----------------------------------------------------------------------------------------------------

 Gao et al., ES&T_2021.pdf
Title:     Direct Prediction of Bioaccumulation of Organic Contaminants in
Authors:   Feng Gao, Yike Shen, Jonathan Brett Sallach, Hui Li, Cun Liu,* and Yuanbo Li*
Abstract:  characterization parameter to describe accumulati

In [156]:
def has_exposure_and_outcome(text):
    text_lower = text.lower()
    exposure_keywords = ["exposure", "exposures"]
    outcome_keywords = ["outcome", "outcomes", "effect", "effects", "impact", "association", "associations"]

    has_exposure = any(word in text_lower for word in exposure_keywords)
    has_outcome = any(word in text_lower for word in outcome_keywords)

    return has_exposure and has_outcome

# Check for exposures and outcomes
for full_path in pdf_files:
    try:
        title, authors, abstract = extract_clean_pdf_info(full_path)

        if has_exposure_and_outcome(abstract):
            print(f"\n📄 {os.path.basename(full_path)}")
            print(f"{'Title:':<10} {title}")
            print(f"{'Authors:':<10} {authors}")
            print(f"{'Abstract:':<10} {abstract[:300]}..." if len(abstract) > 300 else f"{'Abstract:':<10} {abstract}")
            print("🔍 Found both exposure and outcome keywords.")
        else:
            print(f"\n📄 {os.path.basename(full_path)} — Skipped (no exposure/outcome keywords)")

    except Exception as e:
        print(f"\n❌ Failed to read {os.path.basename(full_path)}: {e}")



📄 Prada et al., 2023_eClinMed.pdf — Skipped (no exposure/outcome keywords)

📄 Gao & Shen et al., 2022_JHM.pdf — Skipped (no exposure/outcome keywords)

📄 Gao et al., ES&T_2021.pdf — Skipped (no exposure/outcome keywords)

📄 Gao et al., EI_2022.pdf — Skipped (no exposure/outcome keywords)

📄 Shen et al., EHP_2022.pdf — Skipped (no exposure/outcome keywords)

📄 Campana et al., 2022.pdf — Skipped (no exposure/outcome keywords)

📄 Shen et al 2021_JFP.pdf — Skipped (no exposure/outcome keywords)

📄 Shen et al., 2021_STOTEN.pdf — Skipped (no exposure/outcome keywords)

📄 Shen et al., JHM_2022.pdf — Skipped (no exposure/outcome keywords)

📄 Shen et al., 2019_EI.pdf — Skipped (no exposure/outcome keywords)

📄 Shen et al., 2023_ES&T.pdf
Title:     Cohort Network: A Knowledge Graph toward Data Dissemination
Authors:   Ana Navas-Acien, Andrea A. Baccarelli, and Feng Gao*
Abstract:  on large-scale longitudinal studies to understand the impact of environmental exposures and behavior factors on the

In [164]:
import openai

client = openai.OpenAI(api_key="YOUR_OPENAI_API_KEY_HERE")

prompts = {
    "Exposure (general)": "What is an exposure in environmental health sciences?",
    "Outcome": "What is an outcome in environmental health sciences?",
    "Biomarker": "What is a biomarker in environmental health sciences?",
    "Chemical Exposure": "What is chemical exposure in environmental health sciences?",
    "Physical Exposure": "What is physical exposure in environmental health sciences?",
    "Biomarker Exposure": "What is biomarker exposure in environmental health sciences?",
    "Psychosocial Exposure": "What is psychosocial exposure in environmental health sciences?",
    "Behavior Exposure": "What is behavior exposure in environmental health sciences?",
    "Disease Exposure": "What is disease exposure in environmental health sciences?",
}

def ask_chatgpt(prompt):
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=300
    )
    return response.choices[0].message.content.strip()

# Run prompts
for label, question in prompts.items():
    print(f"\n🔹 {label}")
    print(ask_chatgpt(question))
    print("-" * 100)



🔹 Exposure (general)
An exposure in environmental health sciences refers to the contact of a person or a group with a particular environmental factor or hazard. This could be a chemical substance, a biological agent, a physical condition, or a lifestyle factor. The exposure can occur through various routes such as inhalation, ingestion, skin contact, or even psychological stress. The duration, frequency, and intensity of the exposure can influence the potential health effects.
----------------------------------------------------------------------------------------------------

🔹 Outcome
An outcome in environmental health sciences refers to the result or effect that is caused by exposure to certain environmental factors. This could include physical health effects such as diseases or conditions (like asthma, cancer, or heart disease), mental health effects (like stress or depression), or social effects (like displacement or job loss). These outcomes can be used to measure the impact of 

In [166]:
prompts = {
    "Environmental Health Sciences": "What is chemical exposure in environmental health sciences?",
    "Health Sciences": "What is chemical exposure in health sciences?",
    "Biomedical Sciences": "What is chemical exposure in biomedical sciences?",
    "Public Health": "What is chemical exposure in public health?",
    "Occupational Health": "What is chemical exposure in occupational health?",
    "Toxicology": "What is chemical exposure in toxicology?",
    "Medicine": "What is chemical exposure in medicine?",
}


In [168]:
for label, question in prompts.items():
    print(f"\n🔹 {label}")
    print(ask_chatgpt(question))
    print("-" * 100)



🔹 Environmental Health Sciences
Chemical exposure in environmental health sciences refers to the contact a person has with a chemical, either once or repeatedly over time. This exposure can occur through inhalation, skin contact, or ingestion. The type and severity of health effects that may occur after chemical exposure depend on several factors including the type of chemical, the duration and intensity of exposure, and the individual's health status. Understanding chemical exposure is crucial in environmental health sciences to develop strategies to prevent or reduce harmful exposures and to improve public health.
----------------------------------------------------------------------------------------------------

🔹 Health Sciences
Chemical exposure in health sciences refers to the contact or interaction a person has with a chemical substance, which can potentially lead to harmful health effects. This exposure can occur through inhalation, skin contact, ingestion, or eye contact. Th

In [170]:
!pip install scholarly


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 25.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 25.7 MB/s eta 0:00:00a 0:00:01
  Created wheel for bibtexparser: filename=bibtexparser-1.4.3-py3-none-any.whl size=43550 sha256=a116c264745427eba0c4f2afb0daa3dce293b09ad3e9c7f964988e146fb59168
  Stored in directory: /Users/lauratm/Library/Caches/pip/wheels/1f/7d/e9/1ff2509f13767a55df1279744adfb757f4ab94b2cbe761f56a
  Created wheel for free-proxy: filename=free_proxy-1.1.3-py3-none-any.whl size=6096 sha256=b7c699e2144d63c851852d7cf55f526f8d2bf411a2a8c6fe3ac26042f7dc32de
  Stored in directory: /Users/lauratm/Library/Caches/pip/wheels/b1/5b/61/c93b717842c89aac55d0590076f64041825eb8d89712fca95a
Successfully built bibtexparser free-proxy
  Attempting uninstall: attrs
    Found existing installation: attrs 23.1.0
    Uninstalling attrs-23.1.0:
      Successfully uninstalle

In [172]:
from scholarly import scholarly
import pandas as pd
import time

# List of deduplicated paper titles (from Table S3 – Layer 1)
paper_titles = [
    "Road proximity influences indoor exposures to ambient fine particle mass and components",
    "Associations between air pollution and perceived stress: the veterans administration Normative Aging Study",
    "The effects of daily co-occurrence of affect on older adults' reactivity to health stressors",
    "Do cherished children age successfully? longitudinal findings from the veterans affairs Normative Aging Study",
    "Openness to experience and mortality in men: analysis of trait and facets",
    "Do hassles and uplifts trajectories predict mortality? longitudinal findings from the VA Normative Aging Study",
    "Do stress trajectories predict mortality in older men? longitudinal findings from the VA Normative Aging Study",
    "Healthy psychological functioning and incident coronary heart disease: the importance of self-regulation",
    "Emotional reactivity and mortality: longitudinal findings from the VA Normative Aging Study",
    "The long arm of childhood experiences on longevity: testing midlife vulnerability and resilience pathways",
    "High-fiber foods reduce periodontal disease progression in men aged 65 and older: the veterans affairs Normative Aging Study/dental longitudinal study",
    "Accelerated DNA methylation age and the use of antihypertensive medication among older adults",
    "A western diet pattern is associated with higher concentrations of blood and bone lead among middle-aged and elderly men",
    "Self-reported sleep and beta-amyloid deposition in community-dwelling older adults",
    "Dietary anthocyanin intake and age-related decline in lung function: longitudinal findings from the VA Normative Aging Study",
    "The inflammatory potential of dietary manganese in a cohort of elderly men",
    "Serum tocopherol levels and vitamin e intake are associated with lung function in the Normative Aging Study",
    "Smoking-related DNA methylation is associated with DNA methylation phenotypic age acceleration: the veterans affairs Normative Aging Study",
    "Lifestyle change and high-density lipoprotein change: the us department of veterans affairs Normative Aging Study",
]

# Track seen titles
seen = set()
results = []

# Retrieve each paper's metadata
for title in paper_titles:
    try:
        search = scholarly.search_pubs(title)
        paper = next(search)
        clean_title = paper['bib']['title']

        if clean_title.lower() in seen:
            continue
        seen.add(clean_title.lower())

        results.append({
            "title": clean_title,
            "abstract": paper['bib'].get('abstract', 'Abstract not available'),
            "authors": paper['bib'].get('author', ''),
            "year": paper['bib'].get('pub_year', ''),
            "url": paper.get('pub_url', '')
        })

        time.sleep(2)  # Avoid rate-limiting

    except Exception as e:
        print(f"❌ Failed: {title} — {e}")

# Output result
df = pd.DataFrame(results)
df.to_csv("layer1_google_scholar_abstracts.csv", index=False)
print(df)


❌ Failed: Emotional reactivity and mortality: longitudinal findings from the VA Normative Aging Study — Cannot Fetch from Google Scholar.
❌ Failed: The long arm of childhood experiences on longevity: testing midlife vulnerability and resilience pathways — Cannot Fetch from Google Scholar.
❌ Failed: High-fiber foods reduce periodontal disease progression in men aged 65 and older: the veterans affairs Normative Aging Study/dental longitudinal study — Cannot Fetch from Google Scholar.
❌ Failed: Accelerated DNA methylation age and the use of antihypertensive medication among older adults — Cannot Fetch from Google Scholar.
❌ Failed: A western diet pattern is associated with higher concentrations of blood and bone lead among middle-aged and elderly men — Cannot Fetch from Google Scholar.
❌ Failed: Self-reported sleep and beta-amyloid deposition in community-dwelling older adults — Cannot Fetch from Google Scholar.
❌ Failed: Dietary anthocyanin intake and age-related decline in lung function